## Extracting Checkpoint, Bronze, Silver containers URLS

In [15]:
from common.spark_session import spark
from pyspark.sql.functions import col, current_timestamp

In [16]:
catalog = "sandbox-rey-01"
externalLocation = "loc_sandbox_sblakera"
schema_bronze = "bronze"
schema_silver = "silver"
raw_traffic_table = "raw_traffic"

In [17]:
checkpoint = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "checkpoints"

bronze = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "bronze"

silver = spark.sql(f"""
    DESCRIBE EXTERNAL LOCATION `{externalLocation}`
""").select("url").collect()[0][0] + "silver"

In [18]:
dfBronzeTraffic = (
    spark.readStream
        .table(f"`{catalog}`.`{schema_bronze}`.`{raw_traffic_table}`")
)

## DEDUP

In [19]:
def dedupDF(df):
    print('Deduplicating Dataframe')

    dfDedup = df.dropDuplicates()

    return dfDedup

## NULL REMOVAL

In [20]:
def handleNulls(df, column):
    print("Handling NULL values for string solumns", end='')
    df_string = df.fillna('Unknown', subset=column)
    print("Successfully handled string")

    print('Replacing Null values on Numberic Columns', end='')
    df_numeric = df_string.fillna(0, subset=column)
    print("Successfully handled numeric value")

    print('Filtering out rows with Link_length_km = 0', end='')
    df_clean = df_numeric.filter(col('Link_length_km') > 0)
    print("Successfully filtered zeros")

    return df_clean

## ELECTRIC VEHICLE COUNT

In [21]:
def evCount(df):
    dfEV = df.withColumn('Electric_Vehicles_Count', col('EV_Car') + col('EV_Bike'))
    print("Generated EV count")

    return dfEV

## Motor Vehicle Count

In [22]:
def motorVehicleCount(df):
    dfMotor = df.withColumn('Motor_Vehicles_Count', col('Two_wheeled_motor_vehicles') + col('Cars_and_taxis') + col('Buses_and_coaches') + col('LGV_Type') + col('HGV_Type') + col('Electric_Vehicles_Count'))
    print("Generated Motor Vehicle count")

    return dfMotor

## TRANSFORMED TIME

In [23]:
def createTransformedTime(df):
    print('Creating Transformed Time column : ',end='')
    df_timestamp = df.withColumn('Transformed_Time',
                      current_timestamp()
                      )
    print('Success!!')
    return df_timestamp

## WRITE SILVER TRAFFIC TABLE

In [24]:
def writeTrafficSilverTable(StreamingDF,catalog):
    print('Writing the silver_traffic Data : ',end='') 

    write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',checkpoint+ "/SilverTrafficLoad/Checkpt/")
                .outputMode('append')
                .queryName("SilverTrafficWriteStream")
                .trigger(availableNow=True)
                .toTable(f"`{catalog}`.`{schema_silver}`.`silver_traffic`"))
    
    write_StreamSilver.awaitTermination()
    print(f'Writing `{catalog}`.`{schema_silver}`.`silver_traffic` Success!')

## FUNCTION CALLING

In [25]:
dfBronzeTraffic.schema.names

['Record_ID',
 'Count_point_id',
 'Direction_of_travel',
 'Year',
 'Count_date',
 'hour',
 'Region_id',
 'Region_name',
 'Local_authority_name',
 'Road_name',
 'Road_Category_ID',
 'Start_junction_road_name',
 'End_junction_road_name',
 'Latitude',
 'Longitude',
 'Link_length_km',
 'Pedal_cycles',
 'Two_wheeled_motor_vehicles',
 'Cars_and_taxis',
 'Buses_and_coaches',
 'LGV_Type',
 'HGV_Type',
 'EV_Car',
 'EV_Bike',
 'Extract_Time']

In [26]:
# Remove duplicate rows
dfBronzeDedup = dedupDF(dfBronzeTraffic)

# Replace Null Values
allColumns = dfBronzeDedup.schema.names
dfBronzeNull = handleNulls(dfBronzeDedup, allColumns)

# Get total EV Count
dfEV = evCount(dfBronzeNull)

# Get total Motor Vehicle Count
dfMV = motorVehicleCount(dfEV)

# Add Transformed Time column
dfTransformed = createTransformedTime(dfMV)


Deduplicating Dataframe
Handling NULL values for string solumnsSuccessfully handled string
Replacing Null values on Numberic ColumnsSuccessfully handled numeric value
Filtering out rows with Link_length_km = 0Successfully filtered zeros
Generated EV count
Generated Motor Vehicle count
Creating Transformed Time column : Success!!


In [27]:
# Write to Silver Traffic table

writeTrafficSilverTable(dfTransformed, catalog)

Writing the silver_traffic Data : Writing `sandbox-rey-01`.`silver`.`silver_traffic` Success!


In [28]:
# dfStaticTraffic = spark.read.table(f"`{catalog}`.`{schema_bronze}`.`{raw_traffic_table}`")
# dfStaticMV = motorVehicleCount(evCount(handleNulls(dedupDF(dfStaticTraffic), dfStaticTraffic.schema.names)))
# dfStaticMV.createOrReplaceTempView("ev_preview")

# display(spark.sql("SELECT * FROM ev_preview"))